# 第二章 章节练习

（1）【实验题】请基于下方 SimpleModel 示例代码，完成一下改造：
1. 编写并跑通 Eager 原生单算子执行模式；
2. 对Eager单算子执行模式 和 npugraph_ex 图模式分别采集 NPU Profiling 性能数据，导出性能 Trace 文件；
3. 对比两种模式在device上的耗时， 阐述npugraph_ex 图模式的优化收益。

[profiling使用工具参考](https://docs.pytorch.org/docs/2.12/profiler.html)

In [ ]:
import torch
import torch_npu
from torch_npu.profiler import profile, ProfilerActivity
from torch.profiler import tensorboard_trace_handler

class SimpleModel(torch.nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x, y):
        h1 = torch.add(x, y)
        h2 = torch.add(h1, x)
        h3 = torch.add(h2, y)
        return h3

# 实例化模型，并将模型迁移到NPU设备上
model = SimpleModel().npu()

# 使用 npugraph_ex 后端进行整图编译
opt_model = torch.compile(model,backend="npugraph_ex",fullgraph=True,dynamic=False)

x = torch.randn(2, 64).npu()
y = torch.randn(2, 64).npu()

# ===================== Profiling性能采集 =====================
prof_save_path = "./prof_npugraph"
with profile(
    activities = [ProfilerActivity.CPU, ProfilerActivity.NPU],
    record_shapes = False,
    with_stack = False,
    profile_memory = True,
    schedule = torch_npu.profiler.schedule(wait = 0, active = 2, warmup = 0, repeat = 1),
    on_trace_ready = tensorboard_trace_handler(prof_save_path)
) as prof:
    # 首次执行会触发 Capture + Compile
    out = opt_model(x, y)

print(f"Output shape: {out.shape}")
print(f"Output values:\n{out}")
print(f"\nProfiling数据已导出至目录：{prof_save_path}")

**[参考答案与解析](./answer/02.03_answer.ipynb)**